## Mastering Project Settings

# Unit 8: Granular Environmental Control — Layered JSON Configuration

## Introduction

Welcome back to **Customizing Claude Code for Your Projects**! You've completed the first two units on project instructions, in which we learned how `CLAUDE.md` establishes persistent conventions and standards, and how to control which files it sees using `.claudeignore`. Now, in this third unit, we'll explore a more granular level of workspace customization: **settings files**.

While `CLAUDE.md` defines *what to do* and `.claudeignore` defines *what to see*, settings files control **how Claude operates**—from which model it defaults to, down to how it processes execution permissions. In this lesson, we'll learn to configure behavior at multiple structural levels, creating both team-wide standard matrices and personal environment overlays that shape every single interaction with Claude.

---

## Configuration as Control

Before diving into specific files, let's understand why declarative settings files matter. Every time Claude boots up a session, it must evaluate core operational decisions:

* Which foundational AI model to employ (**Sonnet**, **Haiku**, or **Opus**).
* How to handle user authorization permissions (ask every time, auto-approve specific tools, or outline plans without executing them).
* Which terminal operations to explicitly allow or deny.

Without customization, these decisions follow baseline out-of-the-box behaviors, which might not match your engineering pipeline or security parameters.

Settings files give us declarative control over these choices. Instead of manually running model-switching shortcuts or repeatedly clicking approval boxes for safe commands, we define our environment preferences once, and Claude respects them automatically. More importantly, settings exist at distinct structural levels: some apply globally across your entire operating system, others define absolute team norms for a specific repository, and individual local toggles can override them all. This layered approach lets us balance team consistency with personal flexibility.

---

## The Three Configuration Layers

Claude recognizes three distinct levels of configuration, loaded sequentially from different disk paths:

```text
┌──────────────────────────────────────┐
│  1. User Settings                    │  ◄── Path: ~/.claude/settings.json
│  (Global Host Defaults)              │      Sets your default cross-project preferences.
└──────────────────┬───────────────────┘
                   │
                   ▼
┌──────────────────────────────────────┐
│  2. Project Settings                 │  ◄── Path: [repo]/.claude/settings.json
│  (Shared Team Blueprints)            │      Committed to Git. Dictates project norms.
└──────────────────┬───────────────────┘
                   │
                   ▼
┌──────────────────────────────────────┐
│  3. Local Settings                   │  ◄── Path: [repo]/.claude/settings.local.json
│  (Personal Workspace Overrides)      │      In .gitignore. Overrides all lower configurations.
└──────────────────────────────────────┘

```

### 1. The User Level (`~/.claude/settings.json`)

This configuration lives in your machine's global home directory and establishes your default preference baseline across every project you develop. If you want a specific model to serve as your cross-project engine or always want to force manual code reviews, you declare them here.

### 2. The Project Level (`.claude/settings.json`)

This file is initialized within the root `.claude/` project folder and **should be committed to version control**. It enforces shared standards for every engineer pulling the repository. If a data analysis repository needs to lock down sensitive environment assets or default to a lighter model for cost optimization, those criteria belong here.

### 3. The Local Level (`.claude/settings.local.json`)

Tucked within the project folder but **omitted from Git via `.gitignore**`, this file acts as your personal developer overlay. It allows you to customize workspace behaviors without polluting your team's shared setup. For instance, if your team configuration uses normal execution permissions, but you prefer cautious planning passes, you can toggle that safely inside this file.

---

## Viewing Current Configuration

Let's inspect what is actively configured at your host's user level. We can discover this by asking Claude to expose its baseline settings:

```text
> Show me my current settings
● Read(~/.claude/settings.json)
  ⎿ Read 5 lines
    {
      "permissions": {
        "defaultMode": "default"
      }
    }

```

This JSON output reveals your baseline environment profile. The core configuration houses a `permissions` object tracking a `"defaultMode": "default"` declaration. This signifies that Claude will explicitly pause to prompt you for authorization before executing any mutating filesystem or shell tool calls.

Notice how Claude uses its `Read` tool to safely review the configuration file and streams the lines onto your screen. If you had an implicit global model engine configured, it would surface inside this block as well.

---

## Configuring Project-Specific Defaults

Let's set up a custom workspace profile for a data repository where we want to prioritize cost efficiency and enforce data-access boundaries. We can command Claude to initialize a project configuration directly:

```text
> Create a new file at .claude/settings.json. Configure it to: Set the AI model to "haiku", set the default permission mode to "default", and add a deny array inside the permissions block that blocks access to .env files using these patterns: "Read(./.env)" and "Read(./.env.*)"

● Bash(mkdir -p .claude)
  ⎿ (No content)

● Write(.claude/settings.json)
  ⎿ Wrote 11 lines to .claude/settings.json
    {
      "model": "haiku",
      "permissions": {
        "defaultMode": "default",
        "deny": [
          "Read(./.env)",
          "Read(./.env.*)"
        ]
      }
    }

● Done! I've created the .claude/settings.json file with your requested configuration:
  - Model: Set to "haiku" for faster, lighter processing
  - Permission mode: Set to "default" for the standard safety setting
  - Deny patterns: Blocks Read access to .env and .env.* files

```

### Deconstructing the Project Profile:

* **`"model": "haiku"`**: Locks this specific repository to the rapid, cost-effective Haiku model for handling everyday data queries and routine processing loops. *Note: Project-level model changes take effect when initializing a fresh workspace session.*
* **`"defaultMode": "default"`**: Anchors standard safety rails, requiring explicit manual verification before tools execute mutations.
* **`"deny": [...]`**: Hardcodes security boundaries directly into the runtime context. The ruleset explicitly blocks the agent from loading raw credentials out of root `.env` strings or secondary localized targets like `.env.production`.

---

## Injecting Personal Local Overrides

Shared team setups work perfectly for keeping source patterns uniform, but individual workflows often require personalized execution rules. You can command Claude to spin up a local overlay to safely adjust your workspace permissions:

```text
> I personally want this repo in Plan Mode (local override only)
● Write(.claude/settings.local.json)
  ⎿ Wrote 5 lines to .claude/settings.local.json
    {
      "permissions": {
        "defaultMode": "plan"
      }
    }

```

Claude creates a minimalist `.claude/settings.local.json` file on your filesystem. Because the framework automatically inherits base values from lower configuration tiers, you don't need to re-type model choices or deny patterns here. This file tracks only your active overrides—switching your personal workspace into **Plan Mode** so Claude mocks structural execution paths without running them. Remember to ensure this file is added to your `.gitignore` so your personal environment choices never trigger a branch conflict!

---

## Verifying Active Configuration States

With multiple configuration files laid across your workspace, you can easily audit what settings are actively running by executing the `/status` slash-command:

```text
> /status
Version: 2.1.17
Session ID: b9b5aa03-64de-41b8-b64e-2ed4bd50b984
cwd: /usercode/FILESYSTEM

Model: Default (claude-sonnet-4-5-20250929)
Memory:
Setting sources: User settings, Shared project settings

```

The output gives you clear visibility into your environment layers:

* **Setting sources:** Tells you exactly which files are actively contributing to the environment profile (e.g., combining global host parameters with repository-level rules).
* **Model:** Reflects the active engine driving your current interactive shell window.

---

## Understanding Settings Precedence Hierarchy

When properties overlap across files, flags, or policies, the engine resolves values through a strict, deterministic evaluation hierarchy:

$$\text{Managed Corporate Policies} > \text{CLI Runtime Flags} > \text{Local Overrides } (\text{.local.json}) > \text{Project Settings } (\text{.json}) > \text{User Defaults}$$

### The Complete Chain of Precedence:

1. **Managed Corporate Policies:** Enterprise configurations pushed down by an organization via Claude Teams override all other rules.
2. **CLI Runtime Flags:** Explicit commands entered at launch (e.g., `claude --model opus`) override all file settings for that specific session.
3. **Local Project Overrides (`settings.local.json`):** Highest priority file tier. Allows personal workspace adjustments.
4. **Shared Project Settings (`settings.json`):** Overrides global host profiles to mandate team coding rules.
5. **Global User Defaults (`~/.claude/settings.json`):** The baseline host configuration layer.

---

## Fine-Grained Authorization: Allow and Deny Arrays

Inside the `permissions` object block, the `allow` and `deny` array vectors give you precise control over what operations Claude Code is allowed to run autonomously. Both vectors consume structured **glob-style tool signatures**:

### 🚫 The Deny Array

Operations matching this array are blocked automatically—regardless of what mode your workspace is running in. It acts as an absolute guardrail against accidental modification or sensitive data extraction:

```json
"deny": [
  "Read(./.env)",
  "Bash(rm*)",
  "Write(config/production*)"
]

```

### The Allow Array

Operations declared here bypass interactive confirmation boxes and execute instantly. This is incredibly useful for auto-approving repetitive testing or compilation routines:

```json
"allow": [
  "Read(*.py)",
  "Bash(pytest*)",
  "Write(tests/*)"
]

```

### Pattern Mechanics

The syntax requires the explicit tool name first (`Read`, `Write`, `Bash`, `Edit`), followed immediately by the target file or command wildcard enclosed within parentheses. For example, `Read(*.py)` filters any python asset, while `Bash(pytest*)` safely allows test executions.

> ⚠️ **The Gold Rule:** If a command pattern surfaces inside **both** arrays simultaneously, **the `deny` rule always wins** to protect the integrity of your host environment.

---

## Permission Modes Matrix (`defaultMode`)

How the `allow` and `deny` rulesets behave depends directly on your chosen `defaultMode` setting:

| Config Mode Value | Behavioral Blueprint and Execution Policy |
| --- | --- |
| **`"default"`** | Standard security mode. Pauses to request manual verification for actions, *except* those matching your `allow` list (auto-approved) or your `deny` list (auto-blocked). |
| **`"acceptEdits"`** | Streamlined development mode. Automatically approves localized filesystem file modifications, but forces manual approval for structural shell commands. |
| **`"plan"`** | Pure analysis mode. Claude structures architectural roadmaps and outlines code changes without executing any actions. **`allow` and `deny` lists are ignored.** |
| **`"dontAsk"`** | High-autonomy mode. Automatically executes all file and terminal operations across your environment *unless* they are explicitly blocked by a pattern in your `deny` list. |
| **`"bypassPermissions"`** | Unrestricted access mode. Instantly bypasses every built-in permission guardrail. **Use with extreme caution.** |

---

## Common Configuration Templates

### 🔒 Enterprise Production Profile

Designed for critical environments where you need to block dangerous commands, shield production files, and require explicit approval for tool usage:

```json
{
  "model": "sonnet",
  "permissions": {
    "defaultMode": "default",
    "deny": [
      "Bash(rm*)",
      "Bash(sudo*)",
      "Write(.env*)",
      "Edit(config/production*)"
    ]
  }
}

```

### ⚡ Fast Prototyping Profile

Optimized for rapid iteration, this profile leverages Haiku for speed and auto-approves everyday reading, writing, and test execution tasks:

```json
{
  "model": "haiku",
  "permissions": {
    "defaultMode": "acceptEdits",
    "allow": [
      "Read",
      "Write(*.py)",
      "Bash(python*)",
      "Bash(pytest*)"
    ]
  }
}

```

---

## Summary of Configuration Architecture

| Customization File | Primary Objective | Scope of Influence | Version Control Policy |
| --- | --- | --- | --- |
| **`CLAUDE.md`** | What to build & code conventions | Repo-wide (All Users) | **Commit to Git** |
| **`.claudeignore`** | What files to hide from memory context | Repo-wide (All Users) | **Commit to Git** |
| **`.claude/settings.json`** | How the agent executes & tool modes | Repo-wide (All Users) | **Commit to Git** |
| **`.claude/settings.local.json`** | Personal execution overrides & flags | Machine-wide (Local Only) | **Add to `.gitignore**` |

---

## Conclusion

By mastering JSON settings layers, you gain complete control over how Claude Code operates in your environment. Combining the task boundaries of `CLAUDE.md`, the file filtration of `.claudeignore`, and the granular configuration rules of settings files provides a bulletproof foundation for safe, efficient software development.

Now, let's put these concepts to work and construct your own workspace settings profile in the upcoming lab exercises! Turn the page and let's go.

## Discovering Your Claude Baseline Settings

Before we begin customizing Claude's behavior, let's identify the settings you are currently using. Think of this as checking your "factory defaults" — the baseline configuration Claude uses when no project-specific settings exist.

Ask Claude to display the contents of your user-level settings file located at ~/.claude/settings.json. Immediately afterward, run the /status command to see how those settings translate into your active session configuration.

Pay attention to two key pieces of information:

    Which AI model is currently active in your session (shown in the /status output)
    What permission mode is configured in your settings file (look for the defaultMode field inside the permissions block in settings.json)

These are your global settings — they apply to every project unless you create overrides. Understanding your starting point will make it much easier to see the impact of the customizations you'll create in the next exercises!

Note: If you make changes to settings files during your exploration, you may need to open a new instance of Claude Code for those changes to take effect in your session.

To inspect your active system environment and audit your baseline host configuration profiles, execute the following step-by-step sequence at your terminal prompt:

### Step 1: Read the Global Host Settings

Instruct Claude to inspect your user-level configuration file by entering this prompt at the `>` cursor and pressing **Enter**:

```text
Show me my current settings

```

*(Take a close look at the JSON block that returns—specifically look inside the `permissions` object for the `defaultMode` value, which defines how Claude asks for your confirmation before running tools).*

---

### Step 2: Audit Active Configuration State via `/status`

Once your prompt returns, verify how these file configurations map directly into your active runtime window by executing the status command:

```text
/status

```

* Examine the **`Model`** row to see which foundational engine (such as Sonnet) is processing your session inputs.
* Check the **`Setting sources`** line to confirm that your global user settings are actively fueling the session.

*(Press **`Esc`** to close the status popup window once you have reviewed your environment details).*

---

### Step 3: Conclude and Submit the Exercise

Now that you have successfully verified your active baseline configurations, finalize this introductory tracking module by typing:

```text
/submit

```

By completing this exercise, you have mastered how to audit your global configuration states before rolling out customized team profiles! 🚀

Let's get those exact verification tracking points logged into the lab system. Run these two commands sequentially at your interactive prompt to record the baseline trace:

### Step 1: Read the User-Level Configuration File

Type this exact request at the `>` prompt and press **Enter**:

```text
Show me the contents of ~/.claude/settings.json

```

*(Claude will call its internal `Read` tool to output the raw JSON configuration block containing your `permissions` structure directly onto the screen).*

---

### Step 2: Evaluate the Active Session State

Immediately after the file content displays and the prompt cursor returns, type the status command and press **Enter**:

```text
/status

```

*(Review the popup terminal dashboard to see your active underlying AI model version and to verify that the system states match your `~/.claude/settings.json` file. Press **`Esc`** to dismiss the dialog panel once you're done looking).*

---

### Step 3: Finalize and Submit

With both steps successfully completed in order, conclude your exercise by entering the submission token:

```text
/submit

```

Once the automated lab grader processes your command sequence, you'll be cleared for the next customization module! 🚀

## Creating Your First Project Configuration


Now that you've seen your global settings in action, it's time to take control at the project level!

Ask Claude Code to create a new file at .claude/settings.json in your project directory. Configure it to:

    Set the AI model to "haiku" (a faster, lighter model)
    Create a "permissions" block containing:
        "defaultMode" set to "default" (the default safety setting)
        A "deny" array that blocks access to .env files using these patterns: "Read(./.env)" and "Read(./.env.*)"

Your file should look like this:

JSON
{
  "model": "haiku",
  "permissions": {
    "defaultMode": "default",
    "deny": [
      "Read(./.env)",
      "Read(./.env.*)"
    ]
  }
}

Once you've created the file, run /status to confirm your project settings are active. You should see "Shared project settings" in the Setting sources line of the output, showing that your project configuration is now being recognized alongside your user settings.

Note: You may need to open a new instance of Claude Code for the model change to take effect, as model preferences typically apply when starting a new session.

This is how you establish team-wide standards that everyone on your project will share — perfect for protecting sensitive files and ensuring consistent Claude behavior across your whole team!

Great progress on creating that settings file! It looks like the configuration is spot on. 🚀

However, don't forget the final step: run the /status command to verify that your project settings are active. Give that a try! 🐾

Good effort! It looks like there are two small things to fix:

    Make sure the file is created inside the .claude/ folder (it seems it was saved in the root directory instead).
    To verify your work, type /status directly as a command rather than asking to "open" it like a file.

Give that another shot! 🚀

It looks like you are going through **Unit 8** of the Claude Code interactive training lab! Based on the automated feedback at the bottom of your lesson text, you are almost there, but the grading script flagged two specific issues with your setup.

Let's fix them right now so you can pass this lab.

---

### Step 1: Move the settings file to the correct directory

The lab detected that your `settings.json` file was mistakenly saved in the **root directory** of your project instead of inside the hidden `.claude/` folder.

To fix this, type the following commands directly into your terminal or ask Claude Code to run them for you:

```bash
# 1. Create the hidden .claude directory just in case it doesn't exist
mkdir -p .claude

# 2. Move your settings file into that directory
mv settings.json .claude/settings.json

```

If you need to recreate or edit the file to make sure it's perfect, ensure the path is exactly `.claude/settings.json` and contains this exact JSON:

```json
{
  "model": "haiku",
  "permissions": {
    "defaultMode": "default",
    "deny": [
      "Read(./.env)",
      "Read(./.env.*)"
    ]
  }
}

```

---

### Step 2: Run the verification command correctly

The feedback indicates that instead of executing `/status` as an interactive command, you might have typed something like *"open /status"* or asked Claude to look at a status file.

In Claude Code, `/status` is an immediate **slash command**.

* **What to do:** In your interactive Claude Code terminal session, type exactly this and hit **Enter**:
```text

```



/status

```

---

### What to Verify in the Output
Once you run `/status`, look at the output to confirm it matches the lesson criteria:
* Look at the **Setting sources** line. It must show **"Shared project settings"** (which proves `.claude/settings.json` is being read successfully alongside your global user defaults).
* *(Optional)* If the active model hasn't updated to `haiku` yet, simply exit your Claude Code session and start a new one to force it to reload the new configuration.

Give those two adjustments a shot, and your project settings will be fully active! Let me know if you run into any other errors with the pattern matching or directory structure.

```

## Personal Overrides with Local Settings

Great work setting up project-level configuration! Now let's explore something even more powerful: personal overrides that won't affect your teammates.

Ask Claude Code to create a .claude/settings.local.json file in your project directory. This special file sits at the top of the settings hierarchy, meaning it overrides both global and project settings — but only for you! Here is what to configure:

    Create a "permissions" block containing only:
        "defaultMode" set to "plan" (a mode that lets Claude show you what it plans to do before acting)
    Keep the file minimal — include only the permissions object with defaultMode, nothing else
    Leave the model and deny patterns alone; they'll continue working from your project settings

Your file should look like this:

JSON
{
  "permissions": {
    "defaultMode": "plan"
  }
}

The beauty of local settings is that they're personal. Your .claude/settings.local.json won't (and shouldn't) be committed to version control, so you can experiment with different permission modes without affecting how Claude behaves for your teammates.

Once you've created the file, you'll need to open a new instance of Claude Code for the settings to take effect. Then run /status to verify the configuration. You should see Setting sources now include "Local settings" in addition to the existing sources, confirming that your local override is active and layered on top of the project settings.

This layered system gives you the flexibility to personalize your workflow while maintaining team standards where they matter!

Now you're moving on to the final layer of the configuration sandwich: the personal override! As the lesson layout showed, the `settings.local.json` tier is fantastic because it lets you work in your preferred mode (like **Plan Mode**) without forcing that exact same behavior on the rest of your team.

Here is exactly how to execute this task step-by-step to make sure the lab grading script accepts your work on the first try.

---

### Step 1: Create the Local Settings File

Ask Claude Code to create the file at the exact path `.claude/settings.local.json`. Keep it strictly limited to the `permissions` object override.

The contents of `.claude/settings.local.json` must be exactly:

```json
{
  "permissions": {
    "defaultMode": "plan"
  }
}

```

> 💡 **Why keep it minimal?** Because of Claude Code's deterministic precedence hierarchy, any settings you omit here (like the `"model": "haiku"` and the `.env` deny patterns you set up in the previous step) will gracefully fall back and inherit from your project-level settings.

---

### Step 2: Ensure it is Ignored by Git (Best Practice)

While the lab instructions don't explicitly command you to modify your `.gitignore`, remember the **Golden Rule** from the text: local settings should never be committed to version control. If you have a `.gitignore` file in your root folder, it's always a good habit to ensure `.claude/settings.local.json` is listed there.

---

### Step 3: Restart the Session and Verify

Because changes to core session configurations typically load when the environment initializes, you need to trigger a fresh cycle:

1. **Exit** your current Claude Code session.
2. **Start** a brand-new instance of Claude Code in this project directory.
3. Type the slash command directly into the prompt and hit Enter:
```text
/status

```



```

### What You are Looking For
When the status screen prints out, look closely at the **Setting sources** line. To pass this section, it must dynamically display all active layers:

```text
Setting sources: User settings, Shared project settings, Local settings

```

This confirms that the engine has successfully layered your local **Plan Mode** override directly on top of your shared team blueprint! Give it a spin and see if the lab clears you for the next unit.

## Automating Safe Commands with Allow Patterns

Now that you've mastered the settings hierarchy, it's time to supercharge your workflow with intelligent automation. Let's configure Claude to skip permission requests for operations that you use all the time.

Open a new terminal window and edit your .claude/settings.json file using a text editor. You can use vim .claude/settings.json or ask Claude to edit the file for you. Make two changes inside the "permissions" block:

    Change "defaultMode" from "default" to "acceptEdits"
    Add a new "allow" array containing two patterns: "Bash(ls*)" and "Bash(cat*)"

Your file should look like this when you're done:

JSON
{
  "model": "haiku",
  "permissions": {
    "defaultMode": "acceptEdits",
    "allow": [
      "Bash(ls*)",
      "Bash(cat*)"
    ],
    "deny": [
      "Read(./.env)",
      "Read(./.env.*)"
    ]
  }
}

These allow patterns pre-approve common read-only commands. The acceptEdits mode auto-approves file edits while still checking other operations. Keep your deny array intact — it still blocks .env access even in this mode.

Once saved, you may need to start a new Claude Code instance for the permission changes to take full effect. Return to your Claude session and run /status to confirm that the new mode is active. Then, test your setup: ask Claude to list the files in your directory (ls). It should execute immediately without asking. Next, try asking Claude to remove a file. Since rm isn't in your allow list, Claude should pause and request permission.

This is the sweet spot — you've automated safe, repetitive tasks while maintaining protection against risky operations!

This step is where you really start turning Claude Code into a frictionless development tool! By combining the `"acceptEdits"` mode with an `"allow"` array, you are telling Claude: *"Go ahead and modify local files or run harmless directory checks on your own, but pause and ask me before doing anything major."*

Let’s get this configured so the automation behaves exactly as expected.

---

### Step 1: Update your Project Settings

You need to edit your **shared project settings** file at `.claude/settings.json` (not your local override file). You can open it in your preferred editor (like Neovim or Vim) or have Claude update it for you.

Make sure the file contents strictly match this structure:

```json
{
  "model": "haiku",
  "permissions": {
    "defaultMode": "acceptEdits",
    "allow": [
      "Bash(ls*)",
      "Bash(cat*)"
    ],
    "deny": [
      "Read(./.env)",
      "Read(./.env.*)"
    ]
  }
}

```

> ⚠️ **Important Note on Precedence:** If you still have the `.claude/settings.local.json` file from the previous exercise set to `"defaultMode": "plan"`, it will **override** this new project setting! To test this lesson properly, make sure to temporarily delete, rename, or update your `settings.local.json` so it doesn't mask your new `"acceptEdits"` configuration.

---

### Step 2: Restart and Verify with `/status`

Because session permission policies load when the workspace initializes, reload your session:

1. Exit your current Claude Code terminal session.
2. Start a fresh instance of Claude Code in your project directory.
3. Run the verification slash command:
```text

```



/status

```
   
Verify that the active mode has changed from `default` or `plan` to **`acceptEdits`**.

---

### Step 3: Run the Verification Tests

The grading script for this lab will likely look for you to actively test these newly automated boundaries. Try running these two specific prompts in your Claude Code session to prove the rules work:

1. **Test the Allowed Automation:** Ask Claude: *"List the files in this directory."* 
   * *Expected Behavior:* Claude will execute `ls` instantly in the background and show you the results without popping up an interactive approval box.
2. **Test the Safety Rail:** Ask Claude to do something outside the allowlist, like: *"Create a temporary file and then remove it."*
   * *Expected Behavior:* When Claude tries to run `rm`, it will explicitly pause and ask for your manual permission, because `rm*` is not in your `allow` array.

Once you've run those tests, you'll have successfully hit the sweet spot of secure automation! Let me know if the lab validation passes or if it flags any pattern mismatch.

```